# 00 — behav_utils Example Workflow

Demonstrates the library's three-level convention:
- **Low-level:** `fit_psychometric(stim, ch)` — raw arrays, any source
- **Session-level:** `compute_psychometric(sessions)` — pre-filtered sessions → result dict
- **Plotting:** `plot_psychometric(result)` — result dict → axes

Works with real data (via shared_setup) or synthetic fallback.


## Setup


In [ ]:
%matplotlib inline
from shared_setup import *
from behav_utils.data.synthetic import noisy_psychometric_simulator

experiment, info = load_data()
mode = info['mode']
print(f"Mode: {mode}")
print(f"Animals: {experiment.n_animals}")


In [ ]:
from analysis.grid_search import grid_search_cv
from behav_utils.data.selection import select_sessions
from behav_utils.data.filtering import filter_trials

experiment, _ = load_data()
animal = experiment.get_animal('SS01')
sessions = select_sessions(animal, 'expert_uniform')
clean = filter_trials(sessions)

result = grid_search_cv(clean, 'BE', seed=1)
print(list(result.keys()))

## 1. The Pipeline Pattern

Every analysis follows the same four steps:
select sessions → filter trials → compute result → plot result.


In [ ]:
# Pick an animal
aid = sorted(experiment.animals.keys())[0]
animal = experiment.get_animal(aid)
print(f'Animal: {aid}, {animal.n_sessions} sessions')

# Step 1: Select sessions
sessions = select_sessions(animal, preset='expert_uniform')
print(f'Expert uniform: {len(sessions)} sessions')

# Step 2: Filter trials
clean = filter_trials(sessions)
print(f'After filtering: {len(clean)} sessions')

# Step 3: Compute
psych = compute_psychometric(clean, mode='pooled', n_bootstrap=200)
um = compute_um(clean)
traj = compute_trajectory(clean, stat_names=['accuracy', 'pse'])

# Step 4: Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_psychometric(psych, ax=axes[0], color=PALETTE[0], show_ci=True, title='Psychometric')
plot_um(um, ax=axes[1], title='Update Matrix')
plot_trajectory(traj, 'accuracy', ax=axes[2], title='Accuracy')
plt.suptitle(f'{aid} — Expert Uniform', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 2. Psychometric Modes

`compute_psychometric` supports three modes:
- `'pooled'` — concatenate all trials, single fit
- `'overlay'` — per-session fits
- `'session_mean'` — mean P(B) ± SEM across sessions


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

psych_pooled = compute_psychometric(clean, mode='pooled', n_bootstrap=200)
psych_overlay = compute_psychometric(clean, mode='overlay')
psych_mean = compute_psychometric(clean, mode='session_mean', n_bootstrap=100)

plot_psychometric(psych_pooled, ax=axes[0], color=PALETTE[0], show_ci=True, title='Pooled')
plot_psychometric(psych_overlay, ax=axes[1], title='Overlay')
plot_psychometric(psych_mean, ax=axes[2], color=PALETTE[2], title='Session Mean ± SEM')
plt.tight_layout()
plt.show()


## 3. Comparing Two Conditions

Split the same sessions differently, compute separately, overlay on same axes.


In [ ]:
# Split: early vs late sessions
n_half = len(clean) // 2
early = clean[:n_half]
late = clean[n_half:]

early_psych = compute_psychometric(early, mode='pooled')
late_psych = compute_psychometric(late, mode='pooled')
early_um = compute_um(early)
late_um = compute_um(late)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Psychometric overlay
plot_psychometric(early_psych, ax=axes[0], color=PALETTE[0], label=f'Early ({len(early)}s)')
plot_psychometric(late_psych, ax=axes[0], color=PALETTE[1], label=f'Late ({len(late)}s)')
axes[0].legend(fontsize=9)
axes[0].set_title('Early vs Late')

# UM comparison
plot_um(early_um, ax=axes[1], title='Early UM')
plot_um(late_um, ax=axes[2], title='Late UM')

plt.tight_layout()
plt.show()


## 4. compute_comparison — Full Statistical Test


In [ ]:
comp = compute_comparison(
    early, late,
    n_permutations=500, n_bootstrap=500,
    label_a='Early', label_b='Late',
)

print(f"PSE diff: {comp['diffs']['pse']:.4f}")
print(f"Accuracy diff: {comp['diffs']['accuracy']:.4f}")
if comp.get('perm_p'):
    print(f"Permutation p (PSE): {comp['perm_p'].get('pse', 'N/A')}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_comparison(comp, ax=axes[0], metric='psychometric')
plot_comparison(comp, ax=axes[1], metric='accuracy')
plt.tight_layout()
plt.show()


## 5. Session-Level Stats and Raster


In [ ]:
# Per-session stats
sess = clean[0]
stats = sess.stats(['accuracy', 'pse', 'slope', 'recency', 'win_stay'])
print(f'Stats for {sess.session_id}:')
for k, v in stats.items():
    print(f'  {k:>12s}: {v:.4f}' if isinstance(v, float) else f'  {k:>12s}: {v}')


In [ ]:
# Session raster
raster = compute_session_raster(clean[0])
fig, ax = plot_session_raster(raster, window=20)
plt.show()


## 6. Data Class Convenience Methods

Thin wrappers that call compute_ then plot_ internally.
For quick exploration — use the explicit pipeline for full control.


In [ ]:
# AnimalData wrappers
fig, ax = animal.plot_psychometric(mode='pooled')
plt.show()


In [ ]:
fig, ax = animal.plot_trajectory('accuracy')
plt.show()


In [ ]:
fig, ax = animal.plot_um()
plt.show()


In [ ]:
# SessionData wrappers
fig, ax = clean[0].plot_psychometric()
plt.show()


In [ ]:
fig, ax = clean[0].plot_trials(window=20)
plt.show()


In [ ]:
# Multi-panel overview
fig, axes = animal.plot_overview(stats=['accuracy', 'pse', 'recency'])
plt.show()


## 7. Low-Level Functions (Raw Arrays)

Work with any arrays — model output, synthetic data, numpy.
No SessionData required.


In [ ]:
# Generate synthetic arrays
rng = np.random.default_rng(42)
stim = rng.uniform(-1, 1, 500)
cat = (stim > 0).astype(float)
ch = noisy_psychometric_simulator(stim, cat, rng, sigma=0.20, lapse=0.03)

# Low-level fit
params = fit_psychometric(stim, ch)
print(f"PSE: {params['mu']:.4f}, sigma: {params['sigma']:.4f}")

# Low-level UM
um_arr, _, info = compute_update_matrix(stim, ch, cat)
print(f"UM shape: {um_arr.shape}")

# compute_psychometric also accepts (stim, choices) tuple
result = compute_psychometric((stim, ch), mode='pooled')
fig, ax = plt.subplots(figsize=(5, 4))
plot_psychometric(result, ax=ax, color=PALETTE[0], title='From raw arrays')
plt.show()


## 8. Multiple Animals


In [ ]:
# Compare first two animals
animal_ids = sorted(experiment.animals.keys())[:2]

if len(animal_ids) >= 2:
    fig, ax = plt.subplots(figsize=(6, 4.5))

    for i, aid in enumerate(animal_ids):
        a = experiment.get_animal(aid)
        try:
            sess = select_sessions(a, preset='expert_uniform')
            if len(sess) < 3:
                continue
            c = filter_trials(sess)
            p = compute_psychometric(c, mode='pooled', n_bootstrap=200)
            plot_psychometric(p, ax=ax, color=PALETTE[i], label=aid, show_ci=True)
        except Exception as e:
            print(f'{aid}: {e}')

    ax.legend()
    ax.set_title('Expert Uniform Comparison')
    plt.tight_layout()
    plt.show()
else:
    print('Need at least 2 animals for comparison.')


## Summary

| Level | Function | Input | Output |
|:------|:---------|:------|:-------|
| **Low-level** | `fit_psychometric(stim, ch)` | Arrays | Params dict |
| **Low-level** | `compute_update_matrix(s, ch, cat)` | Arrays | (um, cond, info) |
| **Session-level** | `compute_psychometric(sessions)` | List[SessionData] | Result dict |
| **Session-level** | `compute_um(sessions)` | List[SessionData] | Result dict |
| **Session-level** | `compute_trajectory(sessions, stats)` | List[SessionData] | Result dict |
| **Session-level** | `compute_comparison(a, b)` | List × 2 | Result dict |
| **Plotting** | `plot_psychometric(result)` | Result dict | (fig, ax) |
| **Plotting** | `plot_um(result)` | Result dict | (fig, ax) |
| **Plotting** | `plot_trajectory(result, stat)` | Result dict | (fig, ax) |
| **Plotting** | `plot_comparison(result, metric)` | Result dict | (fig, ax) |

Analysis returns results. Plotting draws results. No exceptions.
